# NinaPro — Results Analysis

Loads all finished MLflow runs from the local `mlruns/` directory (no database needed)
and provides a full comparative analysis with pandas, seaborn and matplotlib.

In [ ]:
import mlflow
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

MLFLOW_EXPERIMENT = "ninapro_emg"   # must match explore_emg.ipynb
MLRUNS_DIR        = "mlruns"        # relative to this notebook

mlflow.set_tracking_uri(MLRUNS_DIR)
client = mlflow.tracking.MlflowClient()

exp = client.get_experiment_by_name(MLFLOW_EXPERIMENT)
if exp is None:
    raise RuntimeError(
        f"Experiment '{MLFLOW_EXPERIMENT}' not found in '{MLRUNS_DIR}'.\n"
        "Run at least one training run in explore_emg.ipynb first."
    )

all_runs = client.search_runs(
    experiment_ids=[exp.experiment_id],
    filter_string="attributes.status = 'FINISHED'",
    max_results=5000,
)
print(f"Found {len(all_runs)} finished runs in experiment '{MLFLOW_EXPERIMENT}'")

In [ ]:
# Build flat DataFrame from runs + per-epoch metrics
records = []
epoch_records = []

for run in all_runs:
    p = run.data.params
    m = run.data.metrics

    row = {
        "run_id":      run.info.run_id,
        "run_name":    run.info.run_name,
        "subject":     int(p.get("subject", -1)),
        "kernels":     p.get("kernels", ""),
        "window_size": int(p.get("window_size", -1)),
        "num_stages":  int(p.get("num_stages", -1)),
        "n_params":    int(p.get("n_params", 0)),
        "epochs":      int(p.get("epochs", 0)),
        "lr":          float(p.get("lr", 0)),
        "weight_decay":float(p.get("weight_decay", 0)),
        "num_classes": int(p.get("num_classes", 0)),
        "train_windows": int(p.get("train_windows", 0)),
        "test_windows":  int(p.get("test_windows", 0)),
        "final_acc":   m.get("final_test_acc", float("nan")),
    }
    records.append(row)

    # Load per-epoch accuracy curves
    for metric_key in ("train_acc", "test_acc"):
        history = client.get_metric_history(run.info.run_id, metric_key)
        for point in history:
            epoch_records.append({
                "run_id":   run.info.run_id,
                "subject":  row["subject"],
                "kernels":  row["kernels"],
                "window_size": row["window_size"],
                "num_stages":  row["num_stages"],
                "metric":   metric_key,
                "epoch":    point.step,
                "value":    point.value,
            })

df       = pd.DataFrame(records).sort_values(["subject", "kernels", "window_size", "num_stages"]).reset_index(drop=True)
df_epoch = pd.DataFrame(epoch_records)

print(f"Runs loaded : {len(df)}")
print(f"Epoch points: {len(df_epoch)}")
df.head()

## 1 — Overall ranking

In [ ]:
pivot = (
    df.groupby(["kernels", "window_size", "num_stages"])["final_acc"]
    .agg(mean="mean", std="std", count="count")
    .reset_index()
    .sort_values("mean", ascending=False)
    .reset_index(drop=True)
)
pivot.index += 1  # rank starts at 1
pivot.index.name = "rank"

# Highlight best row
def highlight_best(s):
    is_best = s == s.max()
    return ["background-color: #d4edda" if v else "" for v in is_best]

pivot.style.apply(highlight_best, subset=["mean"]).format({"mean": "{:.4f}", "std": "{:.4f}"})

## 2 — Accuracy per config (bar chart, mean ± std across subjects)

In [ ]:
pivot_plot = pivot.reset_index().sort_values("mean", ascending=True)
labels = [f"k={r.kernels}\nw={r.window_size} s={r.num_stages}" for r in pivot_plot.itertuples()]
colors = ["#2196F3" if i < len(pivot_plot) - 1 else "#4CAF50" for i in range(len(pivot_plot))]

fig, ax = plt.subplots(figsize=(max(10, len(pivot_plot) * 0.9), 5))
bars = ax.barh(range(len(pivot_plot)), pivot_plot["mean"], xerr=pivot_plot["std"],
               capsize=4, color=colors, edgecolor="white")
ax.set_yticks(range(len(pivot_plot)))
ax.set_yticklabels(labels, fontsize=9)
ax.set_xlabel("Mean Test Accuracy")
ax.set_title("Config ranking — mean ± std across subjects  (green = best)")
ax.xaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.set_xlim(0, 1)
for bar, val in zip(bars, pivot_plot["mean"]):
    ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
            f"{val:.3f}", va="center", fontsize=8)
plt.tight_layout()
plt.show()

## 3 — Per-subject accuracy heatmap

In [ ]:
df["config"] = df.apply(lambda r: f"k={r.kernels} w={r.window_size} s={r.num_stages}", axis=1)
heatmap_data = df.pivot_table(index="subject", columns="config", values="final_acc")

fig, ax = plt.subplots(figsize=(max(8, len(heatmap_data.columns) * 1.2), max(4, len(heatmap_data) * 0.6)))
sns.heatmap(
    heatmap_data, annot=True, fmt=".3f", cmap="YlGn",
    vmin=heatmap_data.values[~np.isnan(heatmap_data.values)].min() - 0.02,
    vmax=1.0, linewidths=0.5, ax=ax,
)
ax.set_title("Test accuracy per subject × config")
ax.set_xlabel("Config"); ax.set_ylabel("Subject")
plt.xticks(rotation=30, ha="right", fontsize=8)
plt.tight_layout()
plt.show()

## 4 — Effect of individual hyperparameters (marginal analysis)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

for ax, col, title in [
    (axes[0], "kernels",     "Kernel config"),
    (axes[1], "window_size", "Window size"),
    (axes[2], "num_stages",  "Num stages"),
]:
    order = df.groupby(col)["final_acc"].mean().sort_values(ascending=False).index.tolist()
    sns.boxplot(data=df, x=col, y="final_acc", order=order, ax=ax, palette="Blues")
    sns.stripplot(data=df, x=col, y="final_acc", order=order, ax=ax,
                  color="black", alpha=0.4, size=4, jitter=True)
    ax.set_title(title)
    ax.set_xlabel(""); ax.set_ylabel("Test Accuracy")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.tick_params(axis="x", rotation=20)

plt.suptitle("Marginal effect of each hyperparameter (all other factors pooled)", y=1.02)
plt.tight_layout()
plt.show()

## 5 — Learning curves for selected configs

In [ ]:
# Show learning curves for the best and worst config (mean across subjects)
best_config  = pivot.iloc[0][["kernels", "window_size", "num_stages"]].to_dict()
worst_config = pivot.iloc[-1][["kernels", "window_size", "num_stages"]].to_dict()

def plot_learning_curves(ax, config: dict, label: str):
    mask = (
        (df["kernels"]     == config["kernels"]) &
        (df["window_size"] == config["window_size"]) &
        (df["num_stages"]  == config["num_stages"])
    )
    run_ids = df.loc[mask, "run_id"].tolist()

    for metric, color, ls in [("train_acc", "#1565C0", "--"), ("test_acc", "#2E7D32", "-")]:
        curves = df_epoch[
            (df_epoch["run_id"].isin(run_ids)) & (df_epoch["metric"] == metric)
        ].groupby("epoch")["value"]

        mean = curves.mean()
        std  = curves.std().fillna(0)
        ax.plot(mean.index, mean.values, color=color, ls=ls,
                label=f"{metric.replace('_', ' ')} (mean)")
        ax.fill_between(mean.index, mean - std, mean + std, alpha=0.15, color=color)

    ax.set_title(f"{label}\nk={config['kernels']} w={config['window_size']} s={config['num_stages']}")
    ax.set_xlabel("Epoch"); ax.set_ylabel("Accuracy")
    ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
    ax.legend(fontsize=8)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4), sharey=True)
plot_learning_curves(ax1, best_config,  "Best config")
plot_learning_curves(ax2, worst_config, "Worst config")
plt.suptitle("Learning curves — mean ± std across subjects", y=1.02)
plt.tight_layout()
plt.show()

## 6 — Per-subject variability

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
subject_order = df.groupby("subject")["final_acc"].mean().sort_values(ascending=False).index.tolist()
sns.boxplot(data=df, x="subject", y="final_acc", order=subject_order, ax=ax, palette="Oranges_r")
sns.stripplot(data=df, x="subject", y="final_acc", order=subject_order, ax=ax,
              color="black", alpha=0.5, size=4, jitter=True)
ax.set_title("Accuracy distribution per subject (across all configs)")
ax.set_xlabel("Subject"); ax.set_ylabel("Test Accuracy")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
plt.tight_layout()
plt.show()

# Table: per-subject summary
print(df.groupby("subject")["final_acc"]
        .agg(mean="mean", std="std", min="min", max="max", count="count")
        .sort_values("mean", ascending=False)
        .to_string())